# Project Dhwani — phase 1 + 2 on Kaggle

Compositional temporal audio grounding: given a recording and a query carrying a
temporal condition, return every interval that satisfies it.

## Before you run anything

Open the panel on the right and set **two** things, or this notebook fails:

1. **Accelerator → GPU T4 x2** (or P100). Cells 5 and 6 need it.
2. **Internet → On.** The notebook clones from GitHub and downloads model weights.

Then use **Save & Run All (Commit)** rather than running interactively. Kaggle
executes the whole notebook server-side for up to 12 hours with the browser
closed, which is what makes this reliable where Colab was not.

## Why Kaggle rather than Colab

Kaggle gives 30 GPU hours a week and 12-hour sessions. `/kaggle/working` is kept
as the notebook's output, so results survive the session ending. Model weights go
to `/kaggle/temp` instead, because `/kaggle/working` is capped at 20 GB and the
weights alone are about 16 GB.

## Runtime

Roughly 3 hours: about 5 minutes of setup and benchmark building, a minute of
CPU experiments, then about 80 minutes per model.

In [ ]:
# 1. Environment. Model weights must NOT land in /kaggle/working, which is capped
# at 20 GB and kept as the notebook output; the weights alone are about 16 GB.
import os, subprocess, shutil

def _pick_cache():
    """Largest writable scratch dir. Kaggle normally has /kaggle/temp, but fall
    back rather than failing three cells later with an opaque disk error."""
    best, best_free = None, 0
    for d in ("/kaggle/temp", "/tmp", "/kaggle/working"):
        try:
            os.makedirs(d, exist_ok=True)
            free = shutil.disk_usage(d).free
            if free > best_free:
                best, best_free = d, free
        except OSError:
            continue
    return best, best_free

cache_root, free = _pick_cache()
os.environ["HF_HOME"] = os.path.join(cache_root, "hf")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print(f"model cache -> {os.environ['HF_HOME']}  ({free / 2**30:.0f} GB free)")
if free < 25 * 2**30:
    print("WARNING: under 25 GB free; the 7B weights plus the dataset may not fit.")

def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True, errors="replace").stdout

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip()
print("gpu:", gpu or "NO GPU - set Accelerator to GPU in the settings panel")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

# A clear failure here beats a confusing one three cells later.
if subprocess.run("timeout 15 git ls-remote https://github.com/Ani2512/mtech-project.git",
                  shell=True, capture_output=True).returncode != 0:
    raise SystemExit("No internet. Turn Internet ON in the notebook settings panel.")
print("internet ok")

In [ ]:
# 2. Code and self-test. Safe to re-run: it updates the code in place and
# leaves data/ and runs/ alone. (An earlier version deleted the whole folder,
# which threw away the 600 MB dataset and the built benchmark with it.)
import os, subprocess, sys

REPO = "/kaggle/working/mtech-project"
os.chdir("/kaggle/working")
if os.path.isdir(os.path.join(REPO, ".git")):
    # data/ and runs/ are gitignored, so a hard reset updates tracked code and
    # leaves everything we have downloaded or computed untouched.
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard",
                    "origin/compositional-temporal-grounding"], check=True)
    print("updated existing checkout, data preserved")
else:
    subprocess.run(["git", "clone", "-q", "-b", "compositional-temporal-grounding",
                    "https://github.com/Ani2512/mtech-project.git", REPO], check=True)
    print("cloned fresh")

os.chdir(f"{REPO}/research_project")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())
print("benchmark present:", os.path.exists("data/esc50/benchmark.jsonl"))

!pip install -q scipy soundfile librosa pyyaml pytest "transformers>=4.52" qwen-omni-utils accelerate bitsandbytes 2>&1 | tail -2
!python -m pytest tests -q -p no:warnings

In [ ]:
# 3. Build the benchmark and score the mock baselines (CPU, about 90 seconds)
!python -m ctag.build_benchmark --source esc50 --n-clips 300 --p-overlap 0.45 --out data/esc50 --esc50-root data/esc50_raw

for mode in ["oracle", "ignore_condition", "first_only"]:
    !python -m ctag.run_zeroshot --model mock:{mode} --bench data/esc50/benchmark.jsonl --out runs/esc50/mock_{mode} > /dev/null
print("mock baselines done")

import json, collections
n, empty = collections.Counter(), collections.Counter()
for line in open("data/esc50/benchmark.jsonl"):
    d = json.loads(line)
    n[d["qtype"]] += 1
    empty[d["qtype"]] += d["expects_empty"]
print("\nqueries by type (share that are rejection queries):")
for t in n:
    print(f"  {t:<14} {n[t]:5d}   {empty[t] / n[t]:.0%}")
print("  total", sum(n.values()))

In [ ]:
# 4. Decomposition ceiling (CPU, about 2 minutes)
#
# The agent answers a conditional query with two plain grounding calls and applies
# the condition itself. Swapping in a perfect grounder, then degrading it, separates
# "are these conditions hard?" from "is the model bad at locating sounds?".
# Perfect grounding must score 1.000; anything less means the agent's composition
# disagrees with the benchmark semantics.
!python -m ctag.run_agent --grounder oracle --bench data/esc50/benchmark.jsonl --timelines data/esc50/timelines.jsonl --out runs/esc50/agent_perfect > /dev/null

# Degrade until PLAIN matches the model's measured grounding quality, then read off
# what perfect condition logic would achieve at that level.
!python -m ctag.run_agent --grounder oracle --jitter 1.5 --drop 0.35 --spurious 0.35 --bench data/esc50/benchmark.jsonl --timelines data/esc50/timelines.jsonl --out runs/esc50/agent_calibrated > /dev/null

import json
for tag in ["agent_perfect", "agent_calibrated"]:
    by = json.load(open(f"runs/esc50/{tag}/summary.json"))["by_type"]["ALL"]
    print(f"{tag:<18} f1@0.5={by['f1@0.5']:.3f}  count_acc={by['count_acc']:.3f}")

In [ ]:
# 5. Qwen2.5-Omni-7B zero-shot, 1200 queries (GPU, about 80 minutes)
# Precision is chosen from the card: fp16 where ~17 GB of weights fit, NF4 4-bit otherwise.
!python -m ctag.run_zeroshot --model qwen2.5-omni --bench data/esc50/benchmark.jsonl --n 1200 --out runs/esc50/qwen25_omni

In [ ]:
# 6. Qwen2-Audio-7B-Instruct zero-shot, 1200 queries (GPU, about 80 minutes)
# Expected to fail almost completely; it is the negative reference.
!python -m ctag.run_zeroshot --model qwen2-audio --bench data/esc50/benchmark.jsonl --n 1200 --out runs/esc50/qwen2_audio

In [ ]:
# 7. Results: failure curve by condition type, and the phase 2 decision
import json, glob, os

runs = {}
for p in sorted(glob.glob("runs/esc50/*/summary.json")):
    s = json.loads(open(p).read())
    runs[s["model"]] = s["by_type"]

TYPES = ["PLAIN", "ORDINAL", "AFTER", "BEFORE", "NEXT_AFTER", "WHILE", "NOT_FOLLOWED", "ABSENT", "ALL"]
COND = ["ORDINAL", "AFTER", "BEFORE", "NEXT_AFTER", "WHILE", "NOT_FOLLOWED"]

for metric in ["f1@0.5", "count_acc", "under_report_rate"]:
    print("\n" + metric)
    print("-" * 141)
    print(f"{'model':<26}" + "".join(f"{t:>13}" for t in TYPES))
    for m in sorted(runs):
        row = ""
        for t in TYPES:
            v = runs[m].get(t, {}).get(metric)
            row += f"{v:>13.3f}" if isinstance(v, (int, float)) else f"{'-':>13}"
        print(f"{m:<26}" + row)

real = [m for m in runs if not m.startswith(("mock", "agent"))]
print("\n" + "=" * 141)
print("PHASE 2 DECISION")
print("=" * 141)
for m in real:
    by = runs[m]
    plain = by.get("PLAIN", {}).get("f1@0.5") or 0.0
    vals = [by[t]["f1@0.5"] for t in COND if by.get(t, {}).get("f1@0.5") is not None]
    cond_mean = sum(vals) / len(vals) if vals else 0.0
    under = by.get("ALL", {}).get("under_report_rate") or 0.0
    parse = by.get("ALL", {}).get("parse_fail_rate") or 0.0
    print(f"\n{m}:  PLAIN f1={plain:.3f}   conditional mean f1={cond_mean:.3f}"
          f"   under-report={under:.0%}   parse-fail={parse:.0%}")
    if parse > 0.3:
        print("  -> Output format is the first problem. Add constrained decoding.")
    elif plain < 0.25:
        print("  -> Fails even PLAIN grounding. PERCEPTION is the bottleneck.")
    elif plain - cond_mean > 0.15:
        print("  -> Grounds the sound but drops the CONDITION. Proceed to LoRA on")
        print("     conditional queries, with the decompose-and-combine agent as comparison.")
    else:
        print("  -> Conditional performance tracks PLAIN; no condition-specific gap.")

In [ ]:
# 8. Statistics: does the PLAIN-vs-conditional gap survive the sample size?
import json, random, statistics, os

COND = ["ORDINAL", "AFTER", "BEFORE", "NEXT_AFTER", "WHILE", "NOT_FOLLOWED"]
path = "runs/esc50/qwen25_omni/predictions.jsonl"
if not os.path.exists(path):
    print("Run cell 5 first.")
else:
    rows = [json.loads(l) for l in open(path)]
    plain = [r["f1@0.5"] for r in rows if r["qtype"] == "PLAIN" and not r["expects_empty"]]
    cond = [r["f1@0.5"] for r in rows if r["qtype"] in COND and not r["expects_empty"]]
    obs = statistics.mean(plain) - statistics.mean(cond)
    print(f"n: PLAIN={len(plain)}  conditional={len(cond)}")
    print(f"mean f1: PLAIN={statistics.mean(plain):.3f}  conditional={statistics.mean(cond):.3f}")
    print(f"observed gap = {obs:.3f}")

    rng = random.Random(0)
    boot = sorted(statistics.mean([rng.choice(plain) for _ in plain])
                  - statistics.mean([rng.choice(cond) for _ in cond]) for _ in range(10000))
    print(f"95% bootstrap CI on the gap: [{boot[250]:.3f}, {boot[9750]:.3f}]")

    pool = plain + cond
    worse = 0
    for _ in range(10000):
        rng.shuffle(pool)
        if statistics.mean(pool[:len(plain)]) - statistics.mean(pool[len(plain):]) >= obs:
            worse += 1
    print(f"permutation p = {worse / 10000:.4f}")
    print("\nThe gap is established only if the CI excludes zero AND p < 0.05.")

---
# Phase 2

Run cells 1 to 4 first. Cells 5 and 6 (the two 1200-query zero-shot runs) are
phase 1 and can be skipped if you only want phase 2.

**Do cell 11 before cell 12.** The trainer has never been executed; a 20-step
smoke test costs two minutes and catches API problems before an 80-minute run.

In [ ]:
# 10. Splits and training data.
# Clip-level, because queries from one clip share audio and a timeline.
!python -m ctag.split --bench data/esc50/benchmark.jsonl --out data/esc50

# plain-ratio 0.6: grounding is the binding constraint (docs/phase2_decomposition.md),
# so most of the signal should be "where does X occur".
for split in ["train", "val"]:
    !python -m ctag.sft_data --bench data/esc50/benchmark_{split}.jsonl --timelines data/esc50/timelines.jsonl --out data/esc50/sft_{split}.jsonl --plain-ratio 0.6
    !python -m ctag.sft_data --bench data/esc50/benchmark_{split}.jsonl --timelines data/esc50/timelines.jsonl --out data/esc50/sft_{split}_tt.jsonl --plain-ratio 0.6 --time-tokens

import json
print(json.load(open("data/esc50/split_stats.json")))

In [ ]:
# 11. SMOKE TEST the trainer: 20 steps.
# train_lora.py has never run. Expect to fix something here rather than 80 minutes in.
# Roughly 6-10 minutes: most of it is the one-time ~16 GB weight download, which
# every later cell then reuses.
#
# Run cell 10 first; this reads the training file it writes.
import os, subprocess, sys

assert os.path.exists("data/esc50/sft_train.jsonl"), "run cell 10 first"
rc = subprocess.run([sys.executable, "-m", "ctag.train_lora",
                     "--data", "data/esc50/sft_train.jsonl",
                     "--out", "/kaggle/temp/smoke",
                     "--max-steps", "20", "--grad-accum", "1"]).returncode
# A bare `print` after a `!` line reports success even when the command failed,
# which is how the first run of this cell claimed the training path worked while
# showing a traceback. Check the exit code instead.
print("SMOKE TEST PASSED - the training path works" if rc == 0
      else f"SMOKE TEST FAILED (exit {rc}) - send me the traceback above")

In [ ]:
# 12. Arm C: QLoRA with plain-text timestamps (about 60-90 minutes)
!python -m ctag.train_lora --data data/esc50/sft_train.jsonl --val data/esc50/sft_val.jsonl --out /kaggle/temp/lora_text --epochs 2
!python -m ctag.run_zeroshot --model qwen2.5-omni --adapter /kaggle/temp/lora_text --bench data/esc50/benchmark_test.jsonl --out runs/esc50/test_lora_text

In [ ]:
# 13. Arm E: QLoRA with atomic timestamp tokens + TEMPO's distance-aware loss.
# Only the output representation differs from arm C, so the comparison is clean.
!python -m ctag.train_lora --data data/esc50/sft_train_tt.jsonl --val data/esc50/sft_val_tt.jsonl --out /kaggle/temp/lora_tt --epochs 2 --time-tokens --time-sigma 0.3 --time-lambda 0.5
!python -m ctag.run_zeroshot --model qwen2.5-omni --adapter /kaggle/temp/lora_tt --bench data/esc50/benchmark_test.jsonl --out runs/esc50/test_lora_tt

In [ ]:
# 14. Baselines on the SAME test split, so every arm is comparable.
# Arm A: direct prompting, untrained.
!python -m ctag.run_zeroshot --model qwen2.5-omni --bench data/esc50/benchmark_test.jsonl --out runs/esc50/test_direct

# Arm B: decompose-and-combine, untrained. Two plain grounding calls per query.
!python -m ctag.run_agent --grounder qwen2.5-omni --bench data/esc50/benchmark_test.jsonl --out runs/esc50/test_agent

# Recall-biased decoding on the untrained model: k samples, keep intervals seen twice.
# Simulations put the optimum at k=5, min_votes=2 (docs/recall_bias.md).
!python -m ctag.run_zeroshot --model qwen2.5-omni --bench data/esc50/benchmark_test.jsonl --out runs/esc50/test_union --samples 5 --min-votes 2 --temperature 0.7

In [ ]:
# 15. Every arm side by side, on the test split only.
import json, glob, os

TYPES = ["PLAIN", "ORDINAL", "AFTER", "BEFORE", "NEXT_AFTER", "WHILE", "NOT_FOLLOWED", "ALL"]
runs = {}
for p in sorted(glob.glob("runs/esc50/test_*/summary.json")):
    s = json.loads(open(p).read())
    runs[os.path.basename(os.path.dirname(p)).replace("test_", "")] = s["by_type"]

for metric in ["f1@0.5", "f_beta"]:
    print("\n" + metric + "   (f_beta weights recall 5.6x, the measured asymmetry)")
    print("-" * (14 + 13 * len(TYPES)))
    print(f"{'arm':<14}" + "".join(f"{t:>13}" for t in TYPES))
    for m in sorted(runs):
        print(f"{m:<14}" + "".join(
            (f"{runs[m][t][metric]:>13.3f}" if isinstance(runs[m].get(t, {}).get(metric), (int, float))
             else f"{'-':>13}") for t in TYPES))

# Arm D: per-type pick between direct and agent, chosen on val, reported on test.
if os.path.exists("runs/esc50/test_direct") and os.path.exists("runs/esc50/test_agent"):
    print()
    !python -m ctag.hybrid --direct runs/esc50/test_direct --agent runs/esc50/test_agent --out runs/esc50/test_hybrid

In [ ]:
# 9. Persist results. /kaggle/working is kept as the notebook's output; the repo
# checkout and the 600 MB dataset are not worth keeping, so copy only what matters.
import shutil, os, glob

out = "/kaggle/working/results"
os.makedirs(out, exist_ok=True)
for src in glob.glob("runs/esc50/*"):
    dst = os.path.join(out, os.path.basename(src))
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)
for doc in glob.glob("docs/*.md"):
    shutil.copy(doc, out)
print("saved to /kaggle/working/results:")
for p in sorted(glob.glob(out + "/*")):
    print("  ", os.path.basename(p))
print("\nDownload these from the notebook's Output tab when the run finishes.")